# 113 — Counter-Assay Informed Delta-ML

Extends nb104 (3-tier delta) with counter-assay information.

Key idea: For a pair (i, j), the "true PXR delta" is:
  delta_pxr = (pEC50_j - pEC50_i) - (null_pEC50_j - null_pEC50_i)

If both i and j have counter-assay measurements, the null delta tells us
how much of the activity difference is off-target vs. PXR-specific.

For compounds without null measurements, use a LGBM null-predictor.
The null-delta feature is added to the existing delta feature set.

In [ ]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test, load_counter
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM_BASE = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                 min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                 reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
print("imports OK")

In [ ]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} r={pr:.4f} rho={sp:.4f}{ca}")
    return m

In [ ]:
# --- Load data and build null-activity predictor ---
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
scaffold_arr = np.array(scaffolds)

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)

PHYS_PROPS = ["mw","logp","tpsa","hbd","hba","rotbonds","fsp3",
               "n_rings","n_aromatic_rings","heavy_atoms","formal_charge"]
print("Computing physchem...", flush=True)
phys_tr = np.array([[p.get(k,0) or 0 for k in PHYS_PROPS]
                     for p in tr["smiles"].map(compute_physchem)], dtype=np.float32)
phys_te = np.array([[p.get(k,0) or 0 for k in PHYS_PROPS]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

# Counter-assay: map to train compounds by SMILES/InChIKey
ctr = load_counter().dropna(subset=["smiles","pec50"])
ctr_map = dict(zip(ctr["smiles"], ctr["pec50"].values))  # smiles -> null_pec50
null_tr_raw = tr["smiles"].map(ctr_map).values  # NaN where not available
n_direct = np.isfinite(null_tr_raw).sum()
print(f"Counter-assay direct matches: {n_direct}/{len(tr)} ({100*n_direct/len(tr):.1f}%)")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)

In [ ]:
# --- Train null-predictor LGBM for imputation ---
print("Training null-pEC50 predictor...", flush=True)
mask_ctr = np.isfinite(null_tr_raw)
X_ctr = X_tr[mask_ctr]
y_ctr = null_tr_raw[mask_ctr]

null_model = lgb.train(
    LGBM_BASE,
    lgb.Dataset(X_ctr, label=y_ctr),
    callbacks=[lgb.log_evaluation(-1)]
)

# Impute: use real null where available, predict otherwise
null_tr = null_tr_raw.copy()
null_tr[~mask_ctr] = null_model.predict(X_tr[~mask_ctr])
null_te = null_model.predict(X_te)  # for test, always predicted

print(f"Null pEC50 range: [{null_tr.min():.2f}, {null_tr.max():.2f}]")
print(f"Corr(pxr, null): {np.corrcoef(y_tr, null_tr)[0,1]:.3f}")
print(f"PXR - Null range: [{(y_tr-null_tr).min():.2f}, {(y_tr-null_tr).max():.2f}]")
pxr_specific = y_tr - null_tr  # "PXR-specific activity"
print(f"PXR-specific std: {pxr_specific.std():.3f}")

In [ ]:
# --- Tanimoto similarity ---
print("Computing pairwise Tanimoto...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

dot_te = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_te / np.maximum(rs_te + rs_tr_v - dot_te, 1e-6)

In [ ]:
# --- Counter-assay-augmented delta features ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_counter_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50,
                              phys_diff, null_anchor, null_query,
                              has_null_anchor, has_null_query):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    # Null delta: (null_query - null_anchor), plus availability flags
    null_delta = (null_query - null_anchor)[:,None]
    null_flags = np.column_stack([has_null_anchor, has_null_query]).astype(np.float32)
    # Stack: 64+64+1+1+11+1+2 = 144 features
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff,
                      null_delta, null_flags])

# Build training pairs
TIERS = {"HIGH": (0.60, 0.90), "MED": (0.45, 0.60), "LOW": (0.35, 0.45)}
i_idx, j_idx = np.where(np.triu(tanimoto_tr > 0.30, k=1))
sim_all = tanimoto_tr[i_idx, j_idx]

def build_tier_data_counter(tier_name, sim_lo, sim_hi):
    if tier_name == "HIGH":
        mask = (sim_all >= sim_lo) & (sim_all <= sim_hi)
    else:
        mask = (sim_all >= sim_lo) & (sim_all < sim_hi)
    ii, jj = i_idx[mask], j_idx[mask]
    sim_ij = sim_all[mask][:,None]
    phys_ij = phys_tr[jj] - phys_tr[ii]
    na_ii = null_tr[ii]; na_jj = null_tr[jj]
    has_ii = mask_ctr[ii].astype(np.float32)
    has_jj = mask_ctr[jj].astype(np.float32)
    F_ij = make_counter_delta_feats(
        fps_tr[ii], fps_tr[jj], sim_ij, y_tr[ii], phys_ij,
        na_ii, na_jj, has_ii, has_jj)
    F_ji = make_counter_delta_feats(
        fps_tr[jj], fps_tr[ii], sim_ij, y_tr[jj], -phys_ij,
        na_jj, na_ii, has_jj, has_ii)
    F = np.vstack([F_ij, F_ji])
    y = np.concatenate([y_tr[jj]-y_tr[ii], y_tr[ii]-y_tr[jj]])
    return F, y, len(ii)

for tier, (lo, hi) in TIERS.items():
    _, _, n = build_tier_data_counter(tier, lo, hi)
    print(f"  {tier} [{lo},{hi}]: {n:,} pairs")
print(f"Feature dim: {build_tier_data_counter('HIGH', 0.60, 0.90)[0].shape[1]}")

In [ ]:
# --- Train counter-augmented tier models ---
DELTA_LGBM = dict(n_estimators=800, num_leaves=63, learning_rate=0.05,
                  min_child_samples=15, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

tier_models = {}
for tier, (lo, hi) in TIERS.items():
    F, y, n_pairs = build_tier_data_counter(tier, lo, hi)
    print(f"Training {tier} counter-delta model on {len(F):,} pairs...", flush=True)
    m = lgb.LGBMRegressor(**DELTA_LGBM)
    m.fit(F, y, callbacks=[lgb.log_evaluation(-1)])
    tier_models[tier] = m
    print(f"  {tier} done.", flush=True)

In [ ]:
K_NEIGHBORS = 10

def predict_counter_delta(fps_q, fps_ref, y_ref, phys_q, phys_ref,
                           null_q, null_ref, has_null_ref, sim_matrix, fallback_preds):
    N = len(fps_q)
    preds = np.full(N, np.nan)
    tier_counts = {t: 0 for t in TIERS}
    tier_counts["fallback"] = 0

    for qi in range(N):
        sim_row = sim_matrix[qi]
        assigned = False

        for tier, (lo, hi) in TIERS.items():
            if tier == "HIGH":
                cand_mask = (sim_row >= lo) & (sim_row <= hi)
            else:
                cand_mask = (sim_row >= lo) & (sim_row < hi)
            cand_idx = np.where(cand_mask)[0]
            if len(cand_idx) == 0:
                continue

            top_k = np.argsort(-sim_row[cand_idx])[:K_NEIGHBORS]
            sel_idx = cand_idx[top_k]
            cand_sims = sim_row[sel_idx]

            fp_q_rep = np.tile(fps_q[qi:qi+1], (len(sel_idx), 1))
            phys_d = phys_q[qi:qi+1] - phys_ref[sel_idx]
            null_q_rep = np.full(len(sel_idx), null_q[qi])
            has_q_rep = np.zeros(len(sel_idx), dtype=np.float32)  # always predicted for query

            F_k = make_counter_delta_feats(
                fps_ref[sel_idx], fp_q_rep, cand_sims[:,None],
                y_ref[sel_idx], phys_d,
                null_ref[sel_idx], null_q_rep,
                has_null_ref[sel_idx], has_q_rep)
            delta_k = tier_models[tier].predict(F_k)
            template_preds = y_ref[sel_idx] + delta_k
            weights = cand_sims ** 2
            preds[qi] = np.average(template_preds, weights=weights)
            tier_counts[tier] += 1
            assigned = True
            break

        if not assigned:
            preds[qi] = fallback_preds[qi]
            tier_counts["fallback"] += 1

    return preds, tier_counts

print("Counter-delta prediction function ready.")

In [ ]:
# --- 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_delta = np.full(len(y_tr), np.nan)
oof_direct = np.full(len(y_tr), np.nan)
has_null_tr = mask_ctr.astype(np.float32)

for fold, (tr_idx, va_idx) in enumerate(splits):
    m_dir = lgb.train(LGBM_BASE, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    preds_d, tc = predict_counter_delta(
        fps_va, fps_ft, y_tr[tr_idx],
        phys_tr[va_idx], phys_tr[tr_idx],
        null_tr[va_idx], null_tr[tr_idx], has_null_tr[tr_idx],
        sim_vf, oof_direct[va_idx])
    oof_delta[va_idx] = preds_d

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_dlt = rae(y_tr[va_idx], oof_delta[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  counter_delta={r_dlt:.4f}  tiers={tc}", flush=True)

m_dir = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_dlt = full_metrics(y_tr, oof_delta,  cliff_pairs, "counter_delta_3tier")

In [ ]:
# Blend sweep & comparison with nb104
best_alpha, best_rae = 1.0, m_dlt["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_delta + (1-alpha)*oof_direct
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    if r < best_rae: best_rae, best_alpha = r, alpha

oof = best_alpha*oof_delta + (1-best_alpha)*oof_direct
print(f"Best alpha={best_alpha:.1f}  OOF RAE={best_rae:.4f}")

nb104_path = DATA_PROCESSED / "oof_delta_similarity_tiers.npy"
nb109_path = DATA_PROCESSED / "oof_delta_ensemble_blend.npy"
if nb104_path.exists():
    print(f"nb104 (3-tier plain):     {rae(y_tr, np.load(nb104_path)):.4f}")
if nb109_path.exists():
    print(f"nb109 (delta blend):      {rae(y_tr, np.load(nb109_path)):.4f}")
print(f"nb113 (counter-augmented): {best_rae:.4f}")

In [ ]:
# --- Final test predictions ---
print("\nFitting final direct LGBM...", flush=True)
m_final = lgb.train(LGBM_BASE, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

print("Running counter-delta on test...", flush=True)
te_delta, te_tc = predict_counter_delta(
    fps_te, fps_tr, y_tr,
    phys_te, phys_tr,
    null_te, null_tr, has_null_tr,
    sim_te_tr, te_direct)
print(f"Test tier usage: {te_tc}")

te_preds = best_alpha*te_delta + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_counter_delta.npy", oof)
np.save(DATA_PROCESSED/"te_oof_counter_delta.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"113_counter_assay_delta.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb113 OOF RAE = {best_rae:.4f} ***")